# VQCFD Gradient and Legacy Variance Analysis

The reusable loading and plotting functions live in `analysis.py`. The first section remains available for old variance-mode results; the new raw-gradient workflow is below.

## Using this notebook

1. Open this notebook from the repository root and run the first setup cell.
2. Restore the selected simulation data to the original relative directory layout.
   The software release does not include the thesis result data.
3. Run the selection/loading cells for the section you need, then its plotting cells.
   Later sections may reuse variables established earlier; follow the notes in each section.
4. Keep the recorded plot settings when reproducing thesis figures. Repeated cells
   preserve alternative plot configurations and are intentional. This is an interactive
   analysis workbook, not a single Run All pipeline.

Saved outputs and execution counters were cleared for the source release. All
original cells, code, comments, run selections, and plot settings are retained.
Create any figure-output directories used by a save cell before executing it.

### Notebook sections

- [Discover legacy variance runs](#discover-legacy-variance-runs)
- [Select runs](#select-runs)
- [Variance plots](#variance-plots)
- [Inspect one run](#inspect-one-run)
- [Discover and select gradient runs](#discover-and-select-gradient-runs)
- [Statistics over all parameters](#statistics-over-all-parameters)
- [Statistics for each parameter](#statistics-for-each-parameter)
- [Configurable plots for overall gradient statistics](#configurable-plots-for-overall-gradient-statistics)
- [Configurable plots for individual parameters](#configurable-plots-for-individual-parameters)
- [Exponential scaling fits over N](#exponential-scaling-fits-over-n)
- [Raw gradients](#raw-gradients)


In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

import analysis as an

an.configure_plots()
RESULTS_ROOT = an.RESULTS_ROOT
RESULTS_ROOT


## Discover legacy variance runs


In [ ]:
variance_runs = an.discover_variance_runs()
an.print_variance_runs(variance_runs)


## Select runs


In [ ]:
# Add (dir_label, label) pairs for a controlled comparison, or leave empty to use all.
RUNS = [
    # ("var", "N2L1"),
    # ("var", "N3L1"),
]

# Each entry replaces its source runs with one variance computed from all raw gradient samples.
VARIANCE_COMBINATIONS = {
    "combined_N10L1": [("var", f"N10L1_{i}") for i in range(1, 5)],
    "combined_N10L2": [("var", f"N10L2_{i}") for i in range(1, 5)],
}

selected = [an.load_variance_run(d, label) for d, label in RUNS] if RUNS else variance_runs
selected = an.apply_variance_combinations(selected, VARIANCE_COMBINATIONS)
an.print_variance_runs(selected)
an.variance_table(selected)


## Variance plots


In [ ]:
an.plot_variance_vs_n(selected, log_y=True);


In [ ]:
an.plot_variance_vs_l(selected, log_y=True);


In [ ]:
an.plot_variance_heatmap(selected);


## Inspect one run


In [ ]:
if selected:
    selected[0]["values"]


## Discover and select gradient runs

Each run contains one raw derivative tensor with axes `(five circuits, parameters, random sweeps)`. Total-cost and contribution gradients are reconstructed from it during analysis.


In [ ]:
gradient_runs = an.discover_gradient_runs()
an.print_gradient_runs(gradient_runs)


In [ ]:
# Add (dir_label, label) pairs for a controlled comparison, or leave empty to use all.
GRADIENT_RUNS = [
    # ("grad", "N2L1"),
    # ("grad", "N3L1"),
]

selected_gradient_runs = (
    [an.load_gradient_run(d, label) for d, label in GRADIENT_RUNS]
    if GRADIENT_RUNS else gradient_runs
)
an.print_gradient_runs(selected_gradient_runs)


## Statistics over all parameters

The unprefixed `mean_gradient`, `mean_gradient_norm`, and `variance` columns describe the reconstructed total-cost gradient. The next 24 columns contain the same three statistics for contributions 1–3 and circuits 1–5. The three prefactor-free contributions are `dc1`, `dc2 - 2*dc1 + dc3`, and `dc4 - dc5`.


In [ ]:
gradient_run_stats = an.gradient_run_table(selected_gradient_runs)
gradient_run_stats


## Statistics for each parameter

For a single scalar parameter, its gradient norm is its absolute value, so `mean_gradient_norm` is the mean absolute gradient.


In [ ]:
gradient_parameter_stats = an.gradient_parameter_table(selected_gradient_runs)
gradient_parameter_stats


## Configurable plots for overall gradient statistics

Choose any column from `gradient_run_stats` for the x-axis. `OVERALL_Y` accepts one column or a list of columns, so the five circuit-component statistics can be drawn together. `OVERALL_FILTERS` accepts any table column with a scalar, a list, or `None`. `OVERALL_GROUP_BY` can be one column, multiple columns, or `None`; include `"circuit"` there when comparing multiple ansatz variants. You can also select line or scatter rendering, logarithmic axes, custom axis and series labels, axis limits, and the figure size. Custom series labels follow the plotted series order and must supply exactly one label per series.


In [ ]:
OVERALL_X = "n"
OVERALL_Y = [f"circuit_{i}_variance" for i in [1,2,4]]
OVERALL_GROUP_BY = "l"#["l","circuit"]  # A column, list of columns, or None
OVERALL_X_LABEL = "N"       # e.g. "Number of qubits"; None uses OVERALL_X
OVERALL_Y_LABEL = "Variance"       # e.g. "Variance"; None uses the selected Y column(s)
OVERALL_SERIES_LABELS = ["Family 1, L = 1","Family 1, L = 4","Family 2, L = 1","Family 2, L = 4","Family 3, L = 1","Family 3, L = 4"]     # e.g. ["Circuit 1", "Circuit 2", "Circuit 4"]

OVERALL_FILTERS = {
    "n": None,                     # e.g. [2, 3, 4]
    "l": [1,4],                      # e.g. [1, 2, 4]
    "circuit": "default",       # e.g. ["multiscale_tree", "uni2"]
    "initial_state": "sine",    # e.g. ["positive_periodic_wave"]
    # Add any other column from gradient_run_stats here.
}

OVERALL_PLOT_KIND = "line"          # "line" or "scatter"
OVERALL_LOG_X = False
OVERALL_LOG_Y = True
OVERALL_X_LIMITS = None           # e.g. (2, 10)
OVERALL_Y_LIMITS = None           # e.g. (1e-8, 1e-2)
OVERALL_PLOT_SIZE = (9, 5)        # (width, height) in inches; None uses the default

overall_plot_data, ax, fig = an.plot_gradient_run_table(
    gradient_run_stats,
    x=OVERALL_X,
    y=OVERALL_Y,
    filters=OVERALL_FILTERS,
    group_by=OVERALL_GROUP_BY,
    kind=OVERALL_PLOT_KIND,
    log_x=OVERALL_LOG_X,
    log_y=OVERALL_LOG_Y,
    x_label=OVERALL_X_LABEL,
    y_label=OVERALL_Y_LABEL,
    series_labels=OVERALL_SERIES_LABELS,
    x_limits=OVERALL_X_LIMITS,
    y_limits=OVERALL_Y_LIMITS,
    figsize=OVERALL_PLOT_SIZE,
)
# overall_plot_data


In [ ]:
# fig.savefig(f"figures/variance/variance_default_default_L2_circuit124.pdf", bbox_inches='tight')

## Configurable plots for individual parameters

`PARAMETER_Y` accepts one statistics column or a list of columns. `PARAMETERS` accepts ordinary indices (`0`, `3`), negative Python-style indices (`-1` is the last parameter), and `"mid"`. The `"mid"` selector is resolved independently for every run as the middle parameter of its middle repeated circuit layer; for even counts, the later of the two middle choices is used. `PARAMETER_FILTERS` accepts any table column with a scalar, a list, or `None`, and grouping by `parameter_selection` compares multiple selectors. You can also select line or scatter rendering, logarithmic axes, custom axis and series labels, axis limits, and the figure size. Custom series labels follow the plotted series order and must supply exactly one label per series.


In [ ]:
PARAMETER_X = "n"
PARAMETER_Y = [f"circuit_{i}_variance" for i in [1,2,4]]
PARAMETERS = [0,2,3, -1, "mid"]
PARAMETER_GROUP_BY = ["l", "parameter_selection"]
PARAMETER_X_LABEL = None       # e.g. "Number of qubits"; None uses PARAMETER_X
PARAMETER_Y_LABEL = None       # e.g. "Variance"; None uses PARAMETER_Y
PARAMETER_SERIES_LABELS = None # e.g. ["First parameter", "Middle parameter"]

PARAMETER_FILTERS = {
    "n": None,                     # e.g. [2, 3, 4]
    "l": None,                     # e.g. [1, 2, 4]
    "circuit": "default",       # e.g. ["multiscale_tree", "uni2"]
    "initial_state": "sine",    # e.g. ["positive_periodic_wave"]
    # Add any other column from gradient_parameter_stats here.
}

PARAMETER_PLOT_KIND = "line"          # "line" or "scatter"
PARAMETER_LOG_X = False
PARAMETER_LOG_Y = True
PARAMETER_X_LIMITS = None           # e.g. (2, 10)
PARAMETER_Y_LIMITS = None           # e.g. (1e-8, 1e-2)
PARAMETER_PLOT_SIZE = (9, 5)        # (width, height) in inches; None uses the default

parameter_plot_data, ax, fig = an.plot_gradient_parameter_table(
    gradient_parameter_stats,
    x=PARAMETER_X,
    y=PARAMETER_Y,
    parameters=PARAMETERS,
    filters=PARAMETER_FILTERS,
    group_by=PARAMETER_GROUP_BY,
    kind=PARAMETER_PLOT_KIND,
    log_x=PARAMETER_LOG_X,
    log_y=PARAMETER_LOG_Y,
    x_label=PARAMETER_X_LABEL,
    y_label=PARAMETER_Y_LABEL,
    series_labels=PARAMETER_SERIES_LABELS,
    x_limits=PARAMETER_X_LIMITS,
    y_limits=PARAMETER_Y_LIMITS,
    figsize=PARAMETER_PLOT_SIZE,
)
# parameter_plot_data


## Exponential scaling fits over N

Choose one positive numeric column or a list of columns from `gradient_run_stats` with `FIT_Y`. Every selected metric is fitted to $y=A\,b^N$ in the same plot, using $\log y=\log A+N\log b$. `FIT_GROUP_BY` controls which independent fits are drawn. For one metric, the legend contains only its grouping values; for multiple metrics, it contains the metric name followed by those grouping values. The fitted $A$, $b$, and log-space $R^2$ remain available in `fit_results`. If several selected runs produce the same N-point within a group, their metric values are averaged before fitting.


In [ ]:
import pandas as pd

FIT_Y = [f"circuit_{i}_variance" for i in [1,2,4]]  # One column or a list of columns
FIT_GROUP_BY = None  # A column, list of columns, or None
FIT_Y_LABEL = None  # e.g. "Variance"; None uses FIT_Y (or "Value" for multiple Ys)

# Optional exact selection by full_label; leave as None to use the filters below.
FIT_RUNS = None  # e.g. ["grad_N2L1", "grad_N3L1", "grad_N4L1"]
FIT_NS = [2,3,4,5,6]  # e.g. [2, 3, 4, 5, 6]
FIT_LS = 2  # e.g. [1, 2, 4]
FIT_INITIAL_STATES = "sine"  # e.g. ["positive_periodic_wave"]
FIT_CIRCUITS = "default"  # e.g. ["multiscale_tree", "uni2"]

fit_data = an.filter_gradient_table(
    gradient_run_stats,
    ns=FIT_NS,
    ls=FIT_LS,
    circuits=FIT_CIRCUITS,
    initial_states=FIT_INITIAL_STATES,
)
if FIT_RUNS is not None:
    fit_run_labels = [FIT_RUNS] if isinstance(FIT_RUNS, str) else FIT_RUNS
    fit_data = fit_data[fit_data["full_label"].isin(fit_run_labels)].copy()
if fit_data.empty:
    raise ValueError("No gradient runs remain after applying the fit filters.")
fit_metrics = [FIT_Y] if isinstance(FIT_Y, str) else list(FIT_Y)
if not fit_metrics:
    raise ValueError("FIT_Y must contain at least one column.")
if any(not isinstance(metric, str) for metric in fit_metrics):
    raise TypeError("Every FIT_Y column must be a string.")
fit_metrics = list(dict.fromkeys(fit_metrics))
missing_metrics = [metric for metric in fit_metrics if metric not in fit_data.columns]
if missing_metrics:
    raise KeyError(f"Unknown FIT_Y columns: {missing_metrics}. Available columns: {list(fit_data.columns)}")

group_columns = (
    [] if FIT_GROUP_BY is None
    else [FIT_GROUP_BY] if isinstance(FIT_GROUP_BY, str)
    else list(FIT_GROUP_BY)
)
missing_group_columns = [column for column in group_columns if column not in fit_data.columns]
if missing_group_columns:
    raise KeyError(f"Unknown FIT_GROUP_BY columns: {missing_group_columns}")
fig, ax = plt.subplots(figsize=(8, 5))
fit_records = []
skipped_fits = []
multiple_metrics = len(fit_metrics) > 1

for metric in fit_metrics:
    grouped_fit_data = (
        fit_data.groupby(group_columns, dropna=False, sort=True)
        if group_columns
        else [((), fit_data)]
    )
    for group_values, group in grouped_fit_data:
        group_values = group_values if isinstance(group_values, tuple) else (group_values,)
        group_metadata = dict(zip(group_columns, group_values))
        points = pd.DataFrame({
            "n": pd.to_numeric(group["n"], errors="coerce"),
            "value": pd.to_numeric(group[metric], errors="coerce"),
        }).dropna()
        points = points[np.isfinite(points["value"]) & (points["value"] > 0)]
        points = points.groupby("n", as_index=False)["value"].mean().sort_values("n")

        if points["n"].nunique() < 2:
            skipped_fits.append({
                "metric": metric,
                **group_metadata,
                "reason": "fewer than two positive N-points",
            })
            continue

        n_values = points["n"].to_numpy(dtype=float)
        y_values = points["value"].to_numpy(dtype=float)
        log_base, log_amplitude = np.polyfit(n_values, np.log(y_values), 1)
        amplitude = float(np.exp(log_amplitude))
        base = float(np.exp(log_base))
        fitted_log_values = log_amplitude + log_base * n_values
        log_residual = np.sum((np.log(y_values) - fitted_log_values) ** 2)
        log_total = np.sum((np.log(y_values) - np.mean(np.log(y_values))) ** 2)
        r2_log = float(1 - log_residual / log_total) if log_total > 0 else np.nan

        label_names = {"l": "L", "n": "N", "initial_state": "init"}
        group_label = ", ".join(
            f"{label_names.get(column, column)}={value}"
            for column, value in group_metadata.items()
        )
        label_parts = [metric] if multiple_metrics else []
        if group_label:
            label_parts.append(group_label)
        line_label = ", ".join(label_parts) or None
        n_fit = np.linspace(n_values.min(), n_values.max(), 200)
        fit_line, = ax.plot(
            n_fit,
            amplitude * base**n_fit,
            label=line_label,
        )
        ax.scatter(n_values, y_values, color=fit_line.get_color(), zorder=3)
        fit_records.append({
            "metric": metric,
            **group_metadata,
            "amplitude_A": amplitude,
            "base_b": base,
            "log_base": float(log_base),
            "r2_log": r2_log,
            "n_points": len(n_values),
        })

ax.set_xlabel("N")
default_y_label = fit_metrics[0] if not multiple_metrics else "Value"
ax.set_ylabel(default_y_label if FIT_Y_LABEL is None else FIT_Y_LABEL)
ax.set_yscale("log")
if (group_columns or multiple_metrics) and ax.has_data():
    ax.legend(fontsize=8)

fig.tight_layout()
fit_results = pd.DataFrame(fit_records)
if skipped_fits:
    print("Skipped fits:")
    for skipped in skipped_fits:
        print("  ", skipped)
fit_results


## Raw gradients

The dictionaries below expose both the saved raw tensors and every reconstructed parameter-by-sweep matrix. For example, use `gradient_components[run_label]["circuit_1"][parameter_index]` or `gradient_components[run_label]["contribution_2"]`.


In [ ]:
gradient_circuit_derivatives = {
    row["full_label"]: an.load_gradient_circuit_derivatives(row)
    for row in selected_gradient_runs
}
gradient_components = {
    row["full_label"]: an.load_gradient_components(row)
    for row in selected_gradient_runs
}
{run: {name: values.shape for name, values in components.items()}
 for run, components in gradient_components.items()}


In [ ]:
len(gradient_components["tests_N2L1"]["total"][0])